# MultiGeometry Stitching (1D + 2D)

When the detector is scanned across multiple angular positions to
extend the q-range coverage, each frame has its own diffraction
geometry.  pyFAI's `MultiGeometry` machinery rebins all the frames
into one common pattern; `ssrl_xrd_tools.integrate.multi` wraps this
in two convenience functions:

* `stitch_1d(images, integrators, ...)` — combined 1D pattern.
* `stitch_2d(images, integrators, ...)` — combined 2D (q, χ) cake.

Both share a single helper `create_multigeometry_integrators` that
takes a base PONI and a per-frame list of detector rotation offsets.

This notebook walks through the full pattern using a stack of images
and a list of `rot1` (in-plane) angles.


## 1. Imports + configuration

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from natsort import natsorted

from ssrl_xrd_tools.io import read_image, load_mask
from ssrl_xrd_tools.integrate import (
    load_poni,
    create_multigeometry_integrators,
    stitch_1d,
    stitch_2d,
)

# ───────── EDIT THESE ─────────
data_dir   = Path('~/data/my_stitched_scan').expanduser()
poni_file  = data_dir / 'calibration.poni'
mask_file  = data_dir / 'mask.edf'
image_glob = '*_scan*.tif'

# Per-image detector rotation offsets (degrees).  Length must match
# the number of images discovered below.  For a 2-circle scan this
# is typically the `del` / `tth` motor stream.
rot1_angles = np.linspace(0.0, 30.0, 11)      # 11 frames, 0° → 30° in 3° steps
rot2_angles = None                            # set to an array if you also vary rot2

# Optional per-image monitor normalisation (i0 / counts / etc.)
normalization = None                          # or e.g. np.array([...])

# Stitched-output parameters
npt_1d       = 2000
npt_rad_2d   = 1000
npt_azim_2d  = 360
unit         = 'q_A^-1'
radial_range = None                           # (qmin, qmax) or None for auto
# ──────────────────────────────

## 2. Load PONI + images + mask

In [ ]:
poni = load_poni(poni_file)
mask = load_mask(mask_file) if mask_file.exists() else None

image_files = natsorted(data_dir.glob(image_glob))
print(f'Found {len(image_files)} frames')
assert len(image_files) == len(rot1_angles), (
    f'image count {len(image_files)} != rot1_angles length {len(rot1_angles)}'
)

images = [read_image(p, mask=mask) for p in image_files]
print(f'Loaded image stack: {len(images)} frames, '
      f'first shape {images[0].shape}')

## 3. Build per-image integrators

`create_multigeometry_integrators` clones the base PONI N times and
applies the per-frame rotation offsets to each.

In [ ]:
integrators = create_multigeometry_integrators(
    poni, rot1_angles, rot2_angles,
)
print(f'Built {len(integrators)} per-image integrators')
print(f'  rot1: {[f"{np.rad2deg(ai.rot1):.2f}°" for ai in integrators[:3]]} ...')

## 4. Stitch into 1D + 2D

In [ ]:
result_1d = stitch_1d(
    images, integrators,
    npt=npt_1d,
    unit=unit,
    method='BBox',
    radial_range=radial_range,
    mask=mask,
    normalization=normalization,
    correctSolidAngle=False,
)

result_2d = stitch_2d(
    images, integrators,
    npt_rad=npt_rad_2d,
    npt_azim=npt_azim_2d,
    unit=unit,
    method='BBox',
    radial_range=radial_range,
    mask=mask,
    normalization=normalization,                # symmetric with stitch_1d
    correctSolidAngle=False,
)

print(f'1D: {result_1d.intensity.shape}, unit={result_1d.unit}')
print(f'2D: {result_2d.intensity.shape}, unit={result_2d.unit}')

## 5. Plot the stitched results

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(result_1d.radial, result_1d.intensity)
ax1.set_xlabel(f'q ({result_1d.unit})')
ax1.set_ylabel('Intensity')
ax1.set_title('Stitched 1D')

im = ax2.pcolormesh(
    result_2d.radial, result_2d.azimuthal, result_2d.intensity.T,
    shading='auto', cmap='viridis',
)
ax2.set_xlabel(f'q ({result_2d.unit})')
ax2.set_ylabel('χ (deg)')
ax2.set_title('Stitched 2D')
plt.colorbar(im, ax=ax2, label='Intensity')

plt.tight_layout()
plt.show()

---

### Notes

- **Per-image normalisation** (`normalization=`) is applied identically
  in both `stitch_1d` and `stitch_2d` — the same array divides each
  image before pyFAI's MultiGeometry bins them together.  Useful when
  you have per-frame monitor counts and want intensities in
  monitor-normalised units.
- For very large stacks where `images` doesn't fit in memory, the
  `StreamingGridder` machinery in `ssrl_xrd_tools.rsm.gridding` shows
  the per-chunk pattern (RSM use case, but the chunking pattern
  generalises).
